# Trace Count v41 screen: parallel retrieval-capacity control

v41 is a single compound architecture control on top of v40. Non-thinking and
Thinking remain two separately initialized-and-trained models. The
256-character prompt, count support 1--5, exact 100 three-character marker
sets, maximum-entropy set/count sampler, separator/no-index trace, partial
count-only output untying, gold-prefix teacher forcing, equal
component-normalized count/trace/structure coefficients, optimizer, 6,000-step
schedule, seed, and inference are unchanged.

Serial depth remains four layers. Residual width grows 256 -> 384 while the
attention head dimension stays exactly 64 (4 -> 6 heads) and the MLP stays 4x
the residual width (1024 -> 1536). This tests whether v40's low free-running
trace/readout stability is a parallel retrieval-capacity bottleneck without
giving Non-thinking additional serial computation. The trace text and targets
are byte-for-byte unchanged. There is no curriculum, scheduled sampling,
auxiliary objective, shared model, checkpoint selection, calibration, or
test-time update.

The preregistered behavioral gate remains Thinking accuracy >= 0.90, minimum
per-count accuracy >= 0.80, trace exact >= 0.90, count spread <= 0.20, and
Thinking-minus-Non-thinking accuracy >= 0.10. NCC and mechanism experiments
run only if the final 6,000-step screen passes.


## 1. Mount Google Drive

In [ ]:
from pathlib import Path

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
DRIVE_READY = False
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        try:
            drive.mount("/content/drive", timeout_ms=300000)
        except ValueError:
            # A stale DriveFS process occasionally survives a Colab reconnect.
            # Clear it and retry once without changing any scientific state.
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            drive.mount(
                "/content/drive", force_remount=True, timeout_ms=300000
            )
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_READY = True
    print("Drive ready:", DRIVE_RESULTS_ROOT)
else:
    print("Local runtime: Drive mount skipped")


## 2. Clone the audited implementation and verify GPU

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

assert DRIVE_READY, "Run the Drive cell first"
REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
REPO_REF = "agent/remove-misplaced-realistic-artifacts"
preferred = Path("/content/Synthetic_CoT_NiaH_Count")
candidates = [Path.cwd(), *Path.cwd().parents, preferred]
repo = next((path.resolve() for path in candidates if (path / "pyproject.toml").exists()), None)
if repo is None:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(preferred)],
        check=True,
    )
    repo = preferred
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", REPO_REF], check=True)
os.chdir(repo)

probe = subprocess.run(
    [sys.executable, "-c", "import numpy,pandas,scipy,matplotlib,seaborn"],
    capture_output=True,
    text=True,
)
if probe.returncode:
    print(probe.stderr[-2000:])
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "--force-reinstall", "numpy==1.26.4", "pandas==2.2.3",
            "scipy==1.13.1", "matplotlib==3.8.4", "seaborn==0.13.2",
        ],
        check=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Scientific ABI repaired. Reconnect and rerun all cells.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
src_root = str(repo / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
os.environ["PYTHONPATH"] = src_root + os.pathsep + os.environ.get("PYTHONPATH", "")

import codecs
import pandas as pd
import torch
import synthetic_counting_v41
from IPython.display import display

def run_streaming(command):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        print(decoder.decode(chunk), end="", flush=True)
    print(decoder.decode(b"", final=True), end="", flush=True)
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print({
    "repo": str(repo),
    "repo_ref": REPO_REF,
    "repo_commit": commit,
    "v28_package": synthetic_counting_v41.__file__,
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
assert torch.cuda.is_available()


## 3. Audit the parallel retrieval-capacity control


In [ ]:
from dataclasses import asdict
from synthetic_counting_v41.config import preset_config
from synthetic_counting_v40.config import preset_config as v40_preset_config
from synthetic_counting_v20.data import V20Vocab, load_corpus_text
from synthetic_counting_v20.model import build_model

VERSION = "v41"
PRESET = "main"
SEEDS = (1234,)
DEVICE = "cuda"
OUT_ROOT = "runs/synthetic_counting_v41"
CHECKPOINT_SYNC_ROOT = DRIVE_RESULTS_ROOT
SKIP_COMPLETED = True
RUN_NAMES = {
    seed: f"v41_count1to5_width384_heads6_steps6000_independent_L256_pool100_seed{seed}" for seed in SEEDS
}
RUN_DIRS = {seed: Path(OUT_ROOT) / RUN_NAMES[seed] for seed in SEEDS}
DRIVE_RUN_DIRS = {seed: CHECKPOINT_SYNC_ROOT / RUN_NAMES[seed] for seed in SEEDS}

planned = preset_config(PRESET, seed=SEEDS[0], device=DEVICE)
baseline = v40_preset_config(PRESET, seed=SEEDS[0], device=DEVICE)
changed_fields = {
    key for key, value in asdict(planned).items()
    if asdict(baseline).get(key) != value
}
assert changed_fields == {"version", "n_head", "n_embd", "n_inner"}, changed_fields
assert planned.count_max_threshold == baseline.count_max_threshold == 5
assert planned.seq_len == baseline.seq_len == 256
assert planned.max_render_len == baseline.max_render_len == 277
assert planned.n_positions == baseline.n_positions == 384
assert planned.needle_pool_size == baseline.needle_pool_size == 100
assert planned.needle_pool_frequency_threshold == baseline.needle_pool_frequency_threshold == 10.0 / 256.0
assert planned.effective_needle_pool_seed == baseline.effective_needle_pool_seed
assert planned.training_count_distribution == baseline.training_count_distribution == "maxent_set_count"
assert planned.batch_size == baseline.batch_size == 128
assert planned.trace_format == baseline.trace_format == "separator"
assert planned.task_output_loss_reduction == baseline.task_output_loss_reduction == "component_normalized"
assert planned.max_steps_for_language_pred == baseline.max_steps_for_language_pred == 1500
assert planned.tie_word_embeddings and baseline.tie_word_embeddings
assert planned.untie_atomic_count_readout and baseline.untie_atomic_count_readout
assert planned.task_output_count_weight == baseline.task_output_count_weight == 8.0
assert planned.task_output_trace_weight == baseline.task_output_trace_weight == 8.0
assert planned.task_output_structure_weight == baseline.task_output_structure_weight == 8.0
assert planned.task_output_scheduled_sampling_max_probability == baseline.task_output_scheduled_sampling_max_probability == 0.0
assert planned.answer_query_contrastive_weight == baseline.answer_query_contrastive_weight == 0.0
assert planned.train_steps == baseline.train_steps == 6000
assert planned.lr == baseline.lr == 3e-4
assert planned.min_lr == baseline.min_lr == 0.0
assert (baseline.n_layer, baseline.n_head, baseline.n_embd, baseline.n_inner) == (4, 4, 256, 1024)
assert (planned.n_layer, planned.n_head, planned.n_embd, planned.n_inner) == (4, 6, 384, 1536)
assert baseline.n_embd // baseline.n_head == planned.n_embd // planned.n_head == 64

text = load_corpus_text()
baseline_vocab = V20Vocab.build(baseline, text)
planned_vocab = V20Vocab.build(planned, text)
baseline_model = build_model(baseline, baseline_vocab, device="cpu").eval()
planned_model = build_model(planned, planned_vocab, device="cpu").eval()
baseline_parameters = baseline_model.parameter_count()
planned_parameters = planned_model.parameter_count()
assert planned_parameters > baseline_parameters
del baseline_model, planned_model

print({
    "controlled_changes_from_v40": sorted(changed_fields),
    "seeds": SEEDS,
    "models": "two independent mode-specific models",
    "trace": "unchanged (<Sep> marker) repeated n times",
    "count_support": "1..5 (unchanged from v40)",
    "serial_depth": "4 layers (unchanged)",
    "parallel_capacity": "4x256 -> 6x384; head dimension remains 64",
    "sampler": "same maximum-entropy feasible set x count family",
    "marker_pool_spec": "exact v40 threshold, size, and seed; set equality checked after prepare",
    "loss": "unchanged v40 schedule and component-normalized CE; count 8, trace 8, structure 8",
    "optimizer_updates": planned.train_steps,
    "baseline_parameters": baseline_parameters,
    "planned_parameters": planned_parameters,
    "checkpoint_selection": False,
    "posthoc_calibration": False,
})

def command_for(seed):
    command = [
        sys.executable, "-u", "-m", "synthetic_counting_v41.run_v41",
        "--preset", PRESET,
        "--device", DEVICE,
        "--seed", str(seed),
        "--train-steps", "6000",
        "--max-steps-for-language-pred", "1500",
        "--n-layer", "4",
        "--n-head", "6",
        "--n-embd", "384",
        "--n-inner", "1536",
        "--checkpoint-every", "100",
        "--recovery-every", "500",
        "--snapshot-shard-every", "500",
        "--eval-every", "500",
        "--ar-eval-every", "1000",
        "--ar-examples-per-count", "2",
        "--permutation-examples-per-count", "1",
        "--eval-examples-per-count", "10",
        "--final-examples-per-count", "50",
        "--phase-head-selection-examples-per-count", "2",
        "--phase-examples-per-count", "1",
        "--out-root", OUT_ROOT,
        "--run-name", RUN_NAMES[seed],
        "--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT),
    ]
    if SKIP_COMPLETED:
        command.append("--skip-completed")
    return command


## 4. Prepare and audit the screening dataset


In [ ]:
for seed in SEEDS:
    run_streaming([*command_for(seed), "--stage", "prepare"])
    print("Prepared:", RUN_DIRS[seed].resolve(), "->", DRIVE_RUN_DIRS[seed])


# Pre-training sampler audit: compare the exact target distribution with the
# retain the v32 low-shortcut target distribution.  This is metadata only and does not inspect test labels.
import numpy as np

from synthetic_counting_v20.pipeline import load_prepared_v20_data
from synthetic_counting_v20.training import _joint_set_count_sampler
audit_vocab = V20Vocab.build(planned, text)
audit_split, audit_pool, _, _ = load_prepared_v20_data(
    planned, audit_vocab, text, RUN_DIRS[SEEDS[0]]
)
joint = _joint_set_count_sampler(planned, text, audit_split, audit_pool)
pivot = joint.plan.pivot(
    index="set_id", columns="count", values="target_probability"
).fillna(0.0)
p = pivot.to_numpy(dtype=float)
ps = p.sum(axis=1, keepdims=True)
pc = p.sum(axis=0, keepdims=True)
mask = p > 0
target_mi_bits = float((p[mask] * np.log2((p / (ps @ pc))[mask])).sum())
target_set_only_bayes = float(p.max(axis=1).sum())
assert target_mi_bits < 0.07
assert target_set_only_bayes < 0.24
print({
    "maxent_target_set_count_MI_bits": target_mi_bits,
    "maxent_target_set_only_Bayes_accuracy": target_set_only_bayes,
    "chance": 0.20,
})


from synthetic_counting_v20.data import build_corpus_split
from synthetic_counting_v20.needle_pool import build_needle_pool
baseline_split = build_corpus_split(baseline, text)
baseline_pool = build_needle_pool(
    baseline, text, baseline_split, baseline_vocab.fingerprint
)
assert [item.characters for item in audit_pool.sets] == [
    item.characters for item in baseline_pool.sets
]
print({"marker_sets_identical_to_v40": True, "marker_set_count": len(audit_pool.sets)})


## 5. Train the two independent models end-to-end


In [ ]:
training_started = time.perf_counter()
for seed in SEEDS:
    print(f"\n===== paired seed {seed} =====", flush=True)
    run_streaming([*command_for(seed), "--stage", "train"])
    sampling = pd.read_csv(RUN_DIRS[seed] / "tables" / "training_sampling_distribution.csv")
    accepted = sampling[sampling["dimension"].eq("accepted_counts")].copy()
    accepted["value"] = accepted["value"].astype(int)
    count_table = accepted.pivot(index="mode", columns="value", values="examples").sort_index(axis=1)
    assert list(count_table.columns) == list(range(1, 6))
    assert count_table.loc["nonthinking"].equals(count_table.loc["thinking"])
    relative_error = (
        count_table.sub(count_table.mean(axis=1), axis=0).abs()
        .div(count_table.mean(axis=1), axis=0)
    )
    assert float(relative_error.to_numpy().max()) < 0.01
    display(pd.read_csv(RUN_DIRS[seed] / "tables" / "final_autoregressive_summary.csv"))
print(f"All paired training: {time.perf_counter() - training_started:.1f} seconds")


## 6. Apply the fixed behavioral gate before NCC/mechanism analysis


In [ ]:
seed_rows = []
for seed in SEEDS:
    summary = pd.read_csv(RUN_DIRS[seed] / "tables" / "final_autoregressive_summary.csv")
    by_count = pd.read_csv(RUN_DIRS[seed] / "tables" / "final_autoregressive_by_count.csv")
    thinking = summary[summary["mode"].eq("thinking")].iloc[0]
    nonthinking = summary[summary["mode"].eq("nonthinking")].iloc[0]
    thinking_counts = by_count[by_count["mode"].eq("thinking")]
    row = {
        "seed": seed,
        "thinking_accuracy": float(thinking.ar_final_accuracy),
        "nonthinking_accuracy": float(nonthinking.ar_final_accuracy),
        "gap": float(thinking.ar_final_accuracy - nonthinking.ar_final_accuracy),
        "thinking_min_count": float(thinking_counts.ar_final_accuracy.min()),
        "thinking_count_spread": float(
            thinking_counts.ar_final_accuracy.max() - thinking_counts.ar_final_accuracy.min()
        ),
        "thinking_trace_exact": float(thinking.trace_exact),
    }
    row["seed_gate"] = bool(
        row["thinking_accuracy"] >= 0.90
        and row["thinking_min_count"] >= 0.80
        and row["thinking_count_spread"] <= 0.20
        and row["thinking_trace_exact"] >= 0.90
        and row["gap"] >= 0.10
    )
    seed_rows.append(row)
behavior = pd.DataFrame(seed_rows)
behavior_gate = bool(behavior.seed_gate.all())
display(behavior)
print({
    "all_seed_behavior_gate": behavior_gate,
    "mean_thinking_accuracy": float(behavior.thinking_accuracy.mean()),
    "mean_nonthinking_accuracy": float(behavior.nonthinking_accuracy.mean()),
    "mean_gap": float(behavior.gap.mean()),
})


## 7. NCC only if the behavioral screen is retained


In [ ]:
if not behavior_gate:
    print("NCC skipped: final behavioral gate failed")
else:
    ncc_rows = []
    for seed in SEEDS:
        output = RUN_DIRS[seed] / "analysis" / "aligned_ncc"
        run_streaming([
            sys.executable, "-u", "scripts/compare_v24_modes_ncc.py",
            "--results-root", str(RUN_DIRS[seed].parent),
            "--output", str(output),
            "--run-prefix", RUN_DIRS[seed].name,
            "--expected-version", VERSION,
            "--device", DEVICE,
            "--discovery-per-label", "10",
            "--confirmation-per-label", "8",
            "--batch-size", "32",
        ])
        frame = pd.read_csv(output / "selected_confirmation_summary.csv")
        frame.insert(0, "seed", seed)
        ncc_rows.append(frame)
    ncc_summary = pd.concat(ncc_rows, ignore_index=True)
    display(ncc_summary[[
        "seed", "comparison_mode", "endpoint", "layer",
        "confirmation_logistic_balanced_accuracy",
        "confirmation_ncc_balanced_accuracy",
    ]])


## 8. Training dynamics and retrieval roles only for a retained screen


In [ ]:
if not behavior_gate:
    print("Mechanism analyses skipped: final behavioral gate failed")
else:
    REFERENCE_SEED = SEEDS[0]
    run_streaming([
        *command_for(REFERENCE_SEED),
        "--stage", "phase,causal,extended,plots",
    ])
    role_table = pd.read_csv(
        RUN_DIRS[REFERENCE_SEED]
        / "analysis" / "extended" / "tables" / "attention_role_dynamics.csv"
    )
    fixed_final = role_table[
        role_table["step"].eq(6000) & role_table["is_fixed_role_head"].eq(1)
    ]
    display(fixed_final[[
        "role", "mode", "layer", "head", "score", "selection_split", "reporting_split"
    ]])


## 9. Verify Drive persistence for the complete screening run


In [ ]:
import json

required = []
for seed in SEEDS:
    drive_run = DRIVE_RUN_DIRS[seed]
    required.extend([
        drive_run / "config.json",
        drive_run / "manifest.json",
        drive_run / "checkpoints" / "rope" / "nonthinking" / "final" / "checkpoint.pt",
        drive_run / "checkpoints" / "rope" / "thinking" / "final" / "checkpoint.pt",
        drive_run / "tables" / "final_autoregressive_summary.csv",
        drive_run / "tables" / "final_autoregressive_by_count.csv",
    ])
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]
assert not missing, missing
for seed in SEEDS:
    manifest = json.loads((DRIVE_RUN_DIRS[seed] / "manifest.json").read_text())
    assert manifest["version"] == "v41"
    assert manifest["stages"]["train"]["status"] == "complete"
print("Drive persistence verified for all seeds:", DRIVE_RUN_DIRS)

if Path("/content").exists():
    print("Experiment and persistence checks complete; disconnecting in 10 seconds.")
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()
